# Text-to-SQL Workshop: Asking a Database Questions in Plain English

In this workshop you will build a **text-to-SQL agent**: a program that takes a question written in everyday language (for example, *"How many employees work at our customers in Dallas?"*), turns it into a database query, runs the query, and explains the result in a sentence.

**By the end of the workshop you will be able to:**
1. Explain what a relational database and SQL are, at the level needed to follow along.
2. Explore an unfamiliar database from Python.
3. Describe how a large language model (LLM) can act as an *agent* that uses tools.
4. Build a text-to-SQL agent with the LangChain library and a locally hosted LLM.
5. Check whether the agent's answer is correct, and recognize where these systems go wrong.

**What you need**
- A GPU job on the cluster, with this notebook running on the `langchain` Jupyter kernel.
- The database file `CRM.db` in the same folder as this notebook.
- Basic Python. No SQL experience is assumed.

**Workshop outline**

| Part | Topic |
|---|---|
| 1 | Background: databases, SQL, and LLM agents |
| 2 | Start the language model |
| 3 | Connect to and explore the database |
| 4 | Meet the agent's tools |
| 5 | Build the text-to-SQL agent |
| 6 | Ask questions and verify the answers |
| 7 | Hands-on exercises |
| 8 | Limitations and key takeaways |

## Part 1. Background

### What is a relational database?
A relational database stores data in **tables**, which look much like spreadsheets: each **row** is one record (for example, one customer company) and each **column** is one attribute of that record (its city, its number of employees, and so on). Tables are linked through **keys**. For example, each row of the `deals` table has an `account_id` column that points to the matching row in the `accounts` table, so we can tell which company a sales deal belongs to.

### What is SQL?
**SQL** (Structured Query Language) is the standard language for asking a relational database questions. A typical query looks like this:

```sql
SELECT SUM(employees_count)
FROM accounts
WHERE billing_city = 'Dallas';
```

Read it as: *"From the `accounts` table, keep only the rows whose billing city is Dallas, then add up their employee counts."* You do not need to write SQL fluently today, but being able to read simple queries will help you check the agent's work.

### Why text-to-SQL?
Many people who need answers from data do not write SQL. Text-to-SQL lets them ask in plain language instead. The challenge is that an LLM knows SQL in general but knows nothing about *your* database: which tables exist, what the columns are called, or how tables link together. It has to look these things up before it can write a correct query.

### What is an LLM agent?
On its own, an LLM only produces text. An **agent** is an LLM placed in a loop where it can call **tools** (ordinary Python functions, such as "list the tables" or "run this query") and read their output. In this notebook the agent follows the **ReAct** pattern, short for *Reason + Act*:

```
Question
  -> Thought:      what should I do next?
  -> Action:       call a tool, e.g. list the tables
  -> Observation:  read what the tool returned
  -> ... repeat Thought / Action / Observation as needed ...
  -> Final Answer
```

### Our data: `CRM.db`
`CRM.db` is a **synthetic** (computer-generated, not real) Customer Relationship Management database, the kind of system a sales team uses to track its customers. It is stored in **SQLite**, a database that lives in a single file and needs no server. It has 36 tables, including:

| Table | What one row represents | Rows |
|---|---|---|
| `accounts` | a customer company (city, revenue, employee count) | 20,000 |
| `deals` | a sales opportunity with an account (amount, stage) | 80,000 |
| `deal_stages` | a step in the sales process, such as `prospecting` or `closed_won` | 7 |
| `industry_types` | an industry, such as Software or Healthcare | 15 |
| `users` | an employee of the sales company | |
| `tickets` | a customer support request | |

## Part 2. Start the language model

We run the LLM on our own GPU using **Ollama**, a program that downloads open-weight models and serves them at a local web address. Because the model runs on the cluster, the questions and the data never leave our system.

The next cell starts the Ollama server in the background. It is specific to our HPC cluster: `module load` makes the Ollama software available. On a personal computer you would install Ollama and run `ollama serve` instead.

In [1]:
import subprocess

server = subprocess.Popen(
    ["bash", "-lc", "module load testing/ollama/0.17.7 gcc/13 && ollama serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

The server needs a few seconds to start. When it is ready, it writes its host name and port number to a file. The next cell waits (up to 60 seconds) until the server answers, then saves its address in `ollama_url`.

In [2]:
import os, time, urllib.request

home_dir = os.getenv("HOME")
job_id = os.getenv("SLURM_JOB_ID")
ollama_dir = os.getenv("OLLAMA_BASE_DIR", home_dir)

# The server writes its port to file a few seconds after starting,
# so re-read the file until the server at that port answers
ollama_url = None
for _ in range(60):
    try:
        with open(f"{ollama_dir}/ollama/host_{job_id}.txt") as f:
            HOST = f.read().strip()
        with open(f"{ollama_dir}/ollama/port_{job_id}.txt") as f:
            PORT = f.read().strip()
        urllib.request.urlopen(f"http://{HOST}:{PORT}/api/tags", timeout=2)
        ollama_url = f"http://{HOST}:{PORT}"
        break
    except Exception:
        time.sleep(1)

if ollama_url is None:
    print("[!] Ollama did not respond after 60 s. Set ollama_url by hand, e.g. ollama_url = 'http://<host>:<port>'")
else:
    print("Ollama is running at:", ollama_url)

Ollama is running at: http://bcm-dgxa100-0005:46172


### Connect to the model

We use **Qwen3 14B**, an open-weight model with 14 billion parameters that is good at writing code, including SQL.

`temperature` controls how random the model's output is. Setting it to `0` makes the model pick its most likely answer every time, which is what we want for SQL: a query should be correct, not creative.

> **Note:** the `Ollama` class used below is marked as deprecated by LangChain and may print a warning. Its replacement is `from langchain_ollama import OllamaLLM`, which takes the same arguments.

In [3]:
from langchain_community.llms import Ollama

llm_model = Ollama(
    base_url=ollama_url,
    model="qwen3:14b",
    temperature=0,
)

/users/tuev/.conda/envs/langchain/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/tmp/ipykernel_619981/3880735847.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm_model = Ollama(


## Part 3. Connect to and explore the database

LangChain's `SQLDatabase` class wraps a database connection so that both we and the agent can use it.

We open `CRM.db` in **read-only mode** (`mode=ro`). This is a safety measure: the agent writes its own SQL, and we do not want a mistaken query to delete or change data. The prompt we write later also tells the agent not to modify anything, but a read-only connection *guarantees* it.

In [4]:
from langchain_community.utilities import SQLDatabase

database = SQLDatabase.from_uri("sqlite:///file:CRM.db?mode=ro&uri=true")
print("Database type (dialect):", database.dialect)

Database type (dialect): sqlite


The **dialect** is the flavor of SQL the database speaks. SQLite, PostgreSQL, and MySQL all use SQL but differ in small details, so the agent needs to know which one it is writing for.

Next, list the tables:

In [5]:
table_names = database.get_usable_table_names()
print(f"There are {len(table_names)} tables:")
print(table_names)

There are 36 tables:
['account_territories', 'accounts', 'business_units', 'calls', 'campaign_members', 'campaigns', 'competitors', 'contacts', 'contracts', 'deal_competitors', 'deal_products', 'deal_stages', 'deals', 'industry_types', 'invoice_lines', 'invoices', 'kb_articles', 'kb_categories', 'lead_sources', 'lead_statuses', 'leads', 'meeting_participants', 'meetings', 'payments', 'price_book_entries', 'price_books', 'products', 'quote_line_items', 'quotes', 'roles', 'tasks', 'territories', 'ticket_priorities', 'ticket_statuses', 'tickets', 'users']


### Look at the structure of one table

The structure of a table (its column names, data types, and links to other tables) is called its **schema**. The cell below prints the schema of `accounts` followed by three example rows. Notice:
- each column has a type, such as `INTEGER`, `TEXT`, or `NUMERIC`;
- `FOREIGN KEY (industry_type_id) REFERENCES industry_types (id)` means the `industry_type_id` column links to the `industry_types` table.

This is exactly the information the agent will read before writing a query.

In [6]:
print(database.get_table_info(["accounts"]))


CREATE TABLE accounts (
	id INTEGER, 
	annual_revenue NUMERIC(14, 2) NOT NULL, 
	billing_city TEXT NOT NULL, 
	billing_country TEXT NOT NULL, 
	billing_postal_code TEXT NOT NULL, 
	billing_state TEXT NOT NULL, 
	billing_street TEXT NOT NULL, 
	employees_count INTEGER NOT NULL, 
	industry_type_id INTEGER NOT NULL, 
	name TEXT NOT NULL, 
	owner_id INTEGER NOT NULL, 
	phone TEXT NOT NULL, 
	shipping_city TEXT NOT NULL, 
	shipping_country TEXT NOT NULL, 
	shipping_postal_code TEXT NOT NULL, 
	shipping_state TEXT NOT NULL, 
	shipping_street TEXT NOT NULL, 
	website TEXT NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(industry_type_id) REFERENCES industry_types (id) DEFERRABLE INITIALLY DEFERRED, 
	FOREIGN KEY(owner_id) REFERENCES users (id) DEFERRABLE INITIALLY DEFERRED, 
	CONSTRAINT ck_accounts_annual_revenue CHECK ("annual_revenue" BETWEEN 500000 AND 900000000), 
	CONSTRAINT ck_accounts_employees_count CHECK ("employees_count" BETWEEN 5 AND 50000)
)

/*
3 rows from accounts table:
id	annual_

### Write SQL yourself first

Before handing the job to an LLM, we run a few queries by hand. The results give us a **ground truth**: a known-correct answer we can compare the agent against later.

The query below shows five accounts whose billing city is Dallas. `pandas` turns the result into a table that is easy to read.

In [7]:
import pandas as pd

query = """
SELECT name, annual_revenue, billing_city, billing_street, employees_count
FROM accounts
WHERE billing_city = 'Dallas'
LIMIT 5
"""

print("Five accounts with a billing city of Dallas")
pd.read_sql(query, database._engine)

Five accounts with a billing city of Dallas


,name,annual_revenue,billing_city,billing_street,employees_count
0,Sipes Inc,7.903028e+08,Dallas,530 Prudence Knolls,2852
1,Casper Group,3.645657e+08,Dallas,358 W Central Avenue,28217
2,"Schulist, Prohaska and DuBuque",3.618331e+08,Dallas,7981 W Church Street,29410
3,"McDermott, O'Kon and Lang",2.505324e+08,Dallas,947 Ferry Road,34184
4,"Kub, Cruickshank and Rogahn",3.462614e+08,Dallas,391 Pagac Crossroad,20209


Now an **aggregate** query, which combines many rows into a single number. `SUM` adds up a column; other aggregates are `COUNT`, `AVG`, `MIN`, and `MAX`.

`database.run` returns the result as text in the form `[(value,)]`: a list of rows, where each row is a tuple of column values. Here there is one row with one value.

In [8]:
query = "SELECT SUM(employees_count) FROM accounts WHERE billing_city = 'Dallas'"
print("Total employees at accounts billed in Dallas:", database.run(query))

Total employees at accounts billed in Dallas: [(21112950,)]


**Try it:** change `'Dallas'` to `'Houston'` in the cell above and run it again. Then try `COUNT(*)` instead of `SUM(employees_count)` to count the accounts rather than add up their employees.

Keep these answers in mind. In Part 6 we will ask the agent the same question and check that it agrees.

## Part 4. Meet the agent's tools

The agent never touches the database directly. Instead it is given four **tools** by LangChain's `SQLDatabaseToolkit`:

| Tool | What it does |
|---|---|
| `sql_db_list_tables` | Returns the names of all tables. |
| `sql_db_schema` | Returns the schema and three sample rows for the tables you name. |
| `sql_db_query_checker` | Asks the LLM to review a query for common mistakes before it runs. |
| `sql_db_query` | Runs a query and returns the result, or the error message if it fails. |

Each tool comes with a short **description**. The LLM reads these descriptions to decide which tool to call, so they work like instructions on a toolbox label.

In [9]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=database, llm=llm_model)
tools = {tool.name: tool for tool in toolkit.get_tools()}

for name, tool in tools.items():
    print(f"{name}:\n    {tool.description}\n")

sql_db_query:
    Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema:
    Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables:
    Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker:
    Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



To see exactly what the agent sees, we can call the tools ourselves. Each tool takes a text input and returns a text output.

In [10]:
print(tools["sql_db_list_tables"].invoke(""))

account_territories, accounts, business_units, calls, campaign_members, campaigns, competitors, contacts, contracts, deal_competitors, deal_products, deal_stages, deals, industry_types, invoice_lines, invoices, kb_articles, kb_categories, lead_sources, lead_statuses, leads, meeting_participants, meetings, payments, price_book_entries, price_books, products, quote_line_items, quotes, roles, tasks, territories, ticket_priorities, ticket_statuses, tickets, users


In [11]:
print(tools["sql_db_schema"].invoke("deal_stages"))


CREATE TABLE deal_stages (
	id INTEGER, 
	is_closed INTEGER NOT NULL, 
	is_won INTEGER NOT NULL, 
	name TEXT NOT NULL, 
	probability INTEGER NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (name), 
	CONSTRAINT ck_deal_stages_name CHECK ("name" IN ('prospecting', 'qualification', 'proposal', 'negotiation', 'closed_won', 'closed_lost', 'on_hold')), 
	CONSTRAINT ck_deal_stages_probability CHECK ("probability" BETWEEN 0 AND 100)
)

/*
3 rows from deal_stages table:
id	is_closed	is_won	name	probability
1	0	0	prospecting	10
2	0	0	qualification	25
3	0	0	proposal	50
*/


In [12]:
print(tools["sql_db_query"].invoke("SELECT name, probability FROM deal_stages"))

[('prospecting', 10), ('qualification', 25), ('proposal', 50), ('negotiation', 75), ('closed_won', 100), ('closed_lost', 0), ('on_hold', 5)]


What happens when a query has a mistake? Below, the column name `name` is misspelled as `nme`.

In [13]:
print(tools["sql_db_query"].invoke("SELECT nme FROM deal_stages"))

Error: (sqlite3.OperationalError) no such column: nme
[SQL: SELECT nme FROM deal_stages]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


Instead of crashing, the tool returns the error message (`no such column: nme`) as ordinary text. This matters: the agent reads the message as an *Observation*, works out what went wrong, and tries a corrected query. This ability to recover from its own errors is a large part of what makes an agent more reliable than a single LLM call.

## Part 5. Build the text-to-SQL agent

### Step 1: Write the instructions (the system prompt)

The **system prompt** is a set of instructions placed in front of every question, like a job description for the agent. Ours tells it to:
- **explore before writing**: list tables, then read the relevant schemas, so it does not guess column names;
- **let the database do the math**: LLMs are unreliable at adding up long lists of numbers, while SQL's `SUM`, `COUNT`, and `AVG` are exact;
- **limit the output**: return at most `top_k` rows so a careless query cannot flood the model with thousands of rows;
- **admit uncertainty**: say so when the data cannot answer the question, rather than inventing an answer;
- **never change data**: read-only, as a second layer of protection on top of the read-only connection.

`{dialect}` and `{top_k}` are placeholders that `.format(...)` fills in with `sqlite` and `200`.

In [14]:
system_prompt = """
You are a SQL assistant that answers questions using a {dialect} database.

Follow these steps in order:
1. List the tables in the database. Never skip this step.
2. Look up the schema of the tables relevant to the question.
3. Write one {dialect} query that answers the question:
   - Run query with the sql_db_query tool.
   - Select only the columns you need; never use SELECT *.
   - Return at most {top_k} rows unless the user asks for a different number.
   - When useful, sort by a relevant column so the most informative rows come first.
4. Check the query for errors before running it. If it fails, fix it and retry.
5. Answer the question in plain language based only on the query results.
   If the data cannot answer the question, say so instead of guessing.

Rules:
- For totals, counts, averages, minimums, or maximums, compute the result in SQL
  with SUM, COUNT, AVG, MIN, or MAX. Never calculate from returned rows yourself.
- The row limit applies only to queries that list rows, not to aggregate queries.
- Read-only access: never run INSERT, UPDATE, DELETE, DROP, ALTER, or any other
  statement that changes the database.
""".format(dialect=database.dialect, top_k=200)

### Step 2: Create the agent

`create_sql_agent` combines the LLM, the tools, and the system prompt into an agent. The arguments:

| Argument | Meaning |
|---|---|
| `prefix` | Our system prompt, placed before the tool descriptions and the question. |
| `agent_type` | `ZERO_SHOT_REACT_DESCRIPTION`: *zero-shot* means the model gets no worked examples, *ReAct* is the Thought/Action/Observation loop from Part 1, and *description* means it picks tools by reading their descriptions. |
| `max_iterations` | The most Thought/Action/Observation rounds allowed before the agent gives up, so it cannot loop forever. |
| `verbose` | Print every step, so we can watch the agent think. |
| `handle_parsing_errors` | If the LLM's reply is not in the expected format, send it a reminder instead of crashing. |
| `return_intermediate_steps` | Keep a record of every tool call, so we can inspect the SQL the agent ran. |

This is LangChain's older ("classic") agent interface. It is simple to read, which makes it good for learning. For new projects LangChain recommends building agents with LangGraph; `langgraph_agent.py` in this repository shows that approach.

In [15]:
from langchain_community.agent_toolkits import create_sql_agent
from langchain_classic.agents.agent_types import AgentType

agent = create_sql_agent(
    llm=llm_model,
    toolkit=toolkit,
    prefix=system_prompt,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    max_iterations=15,
    verbose=True,
    agent_executor_kwargs={
        "handle_parsing_errors": True,
        "return_intermediate_steps": True,
    },
)

## Part 6. Ask questions and verify the answers

### Your first question

We ask the same question we answered by hand in Part 3. Each Thought/Action/Observation round is a separate call to the LLM, so expect this to take a while. `%%time` reports how long it took.

While it runs, read the printed trace:
- **Action** is the tool the agent chose, and **Action Input** is what it passed to that tool (a table name, or a SQL query);
- the text that follows is the **Observation**, the tool's output;
- **Final Answer** is the agent's reply to you.

In [16]:
%%time
question = "How many employees in total work at accounts whose billing city is Dallas?"
result = agent.invoke({"input": question})



> Entering new SQL Agent Executor chain...
Thought: I should start by listing the tables in the database to identify which ones might be relevant.
Action: sql_db_list_tables
Action Input: account_territories, accounts, business_units, calls, campaign_members, campaigns, competitors, contacts, contracts, deal_competitors, deal_products, deal_stages, deals, industry_types, invoice_lines, invoices, kb_articles, kb_categories, lead_sources, lead_statuses, leads, meeting_participants, meetings, payments, price_book_entries, price_books, products, quote_line_items, quotes, roles, tasks, territories, ticket_priorities, ticket_statuses, tickets, usersThought: I need to check the schema of the 'accounts' and 'users' tables to confirm the relevant columns.
Action: sql_db_schema
Action Input: accounts, users
CREATE TABLE accounts (
	id INTEGER, 
	annual_revenue NUMERIC(14, 2) NOT NULL, 
	billing_city TEXT NOT NULL, 
	billing_country TEXT NOT NULL, 
	billing_postal_code TEXT NOT NULL, 
	billing_

In [17]:
print(result["output"])

The total number of employees working at accounts with a billing city of Dallas is 21,112,950.


### Verify the answer

An agent can sound confident and still be wrong, so **never accept an answer without checking it**. Two checks:
1. **Compare with the ground truth.** In Part 3 our own query returned 21,112,950. Does the agent agree?
2. **Read the SQL it ran.** Even when you have no ground truth, you can read the query and ask: is this the right table? The right filter? The right aggregate?

`result["intermediate_steps"]` holds every tool call. The cell below prints only the SQL queries and their results.

In [18]:
for action, observation in result["intermediate_steps"]:
    if action.tool == "sql_db_query":
        print("SQL the agent ran:")
        print(action.tool_input)
        print("Result:", observation, "\n")

SQL the agent ran:
SELECT SUM(employees_count) AS total_employees FROM accounts WHERE billing_city = 'Dallas';
Result: [(21112950,)] 



### A helper for the rest of the workshop

The function `ask` wraps the last few steps: it sends a question to the agent, then prints the final answer and the SQL behind it.

In [19]:
def ask(question):
    result = agent.invoke({"input": question})
    print("\n" + "=" * 70)
    print("QUESTION:", question)
    for action, observation in result["intermediate_steps"]:
        if action.tool == "sql_db_query":
            print("\nSQL:\n" + str(action.tool_input).strip())
            print("RESULT:", observation)
    print("\nANSWER:", result["output"])
    return result

### A question that needs two tables

The `accounts` table stores each company's industry only as a number (`industry_type_id`); the industry *names* live in the `industry_types` table. To answer the question below, the agent has to **join** the two tables, that is, match each account to its industry using the shared key.

The correct answer is Media (5,205 accounts), Legal (2,101), and Logistics (1,715). Check whether the agent's SQL contains a `JOIN`.

In [20]:
%%time
result = ask("Which 3 industries have the most accounts? Show the industry name and the number of accounts.")



> Entering new SQL Agent Executor chain...
Thought: I should start by listing the tables in the database to identify which ones might contain industry and account information.
Action: sql_db_list_tables
Action Input: account_territories, accounts, business_units, calls, campaign_members, campaigns, competitors, contacts, contracts, deal_competitors, deal_products, deal_stages, deals, industry_types, invoice_lines, invoices, kb_articles, kb_categories, lead_sources, lead_statuses, leads, meeting_participants, meetings, payments, price_book_entries, price_books, products, quote_line_items, quotes, roles, tasks, territories, ticket_priorities, ticket_statuses, tickets, usersThought: I need to check the schema of the accounts and industry_types tables to find the correct column names for joining and selecting data.
Action: sql_db_schema
Action Input: accounts, industry_types
CREATE TABLE accounts (
	id INTEGER, 
	annual_revenue NUMERIC(14, 2) NOT NULL, 
	billing_city TEXT NOT NULL, 
	bil

## Part 7. Hands-on exercises

Work through the exercises below with a partner. For each one, run `ask(...)`, then compare the agent's answer with the expected answer (click **Show answer**). If they differ, read the agent's SQL and work out where it went wrong.

### Exercise 1: Counting with a lookup
How many deals have been won? *Hint: the stage of a deal is stored as a number in `deals.stage_id`; the stage names, such as `closed_won`, are in `deal_stages`.*

<details><summary>Show answer</summary>

7,252 deals.

```sql
SELECT COUNT(*)
FROM deals d JOIN deal_stages s ON d.stage_id = s.id
WHERE s.name = 'closed_won';
```
</details>

In [ ]:
result = ask("How many deals have been won?")

### Exercise 2: Adding up values
What is the total amount of money in won deals?

<details><summary>Show answer</summary>

530,986,337.33 (US dollars).

```sql
SELECT SUM(d.amount)
FROM deals d JOIN deal_stages s ON d.stage_id = s.id
WHERE s.name = 'closed_won';
```
</details>

In [ ]:
result = ask("What is the total amount of money in won deals?")

### Exercise 3: Ranking
Which five billing cities have the most accounts? Write your own wording of the question in the cell below.

<details><summary>Show answer</summary>

New York (3,303), Los Angeles (2,124), Chicago (1,567), Dallas (1,238), Houston (1,184).

```sql
SELECT billing_city, COUNT(*) AS n
FROM accounts
GROUP BY billing_city
ORDER BY n DESC
LIMIT 5;
```
</details>

In [ ]:
result = ask("...")  # replace ... with your question

### Exercise 4: An ambiguous question (discussion)
Ask: *"Who is our best salesperson?"*

"Best" has no single meaning. It could mean the most money won, the most deals won, or the highest win rate. Look at the SQL: which definition did the agent choose? Did it tell you it made a choice? Then rewrite the question so that only one answer is possible.

<details><summary>Show one possible answer</summary>

If "best" means the most money in won deals, the answer is Lenore VonRueden with 15,374,189.39.

```sql
SELECT u.first_name, u.last_name, SUM(d.amount) AS won_amount
FROM deals d
JOIN deal_stages s ON d.stage_id = s.id
JOIN users u ON d.owner_id = u.id
WHERE s.name = 'closed_won'
GROUP BY u.id
ORDER BY won_amount DESC
LIMIT 1;
```
</details>

In [ ]:
result = ask("Who is our best salesperson?")

### Exercise 5: Safety check
Ask the agent to change the data: *"Delete all support tickets."*

There are two layers of protection. The system prompt tells the agent to refuse, and the read-only connection blocks any write even if the agent ignores the prompt. Watch the trace: did the agent refuse, or did it try and receive an `attempt to write a readonly database` error? Why is it a bad idea to rely on the prompt alone?

In [ ]:
result = ask("Delete all support tickets.")

### Exercise 6 (optional): Change the instructions
Edit `system_prompt` in Part 5, for example by deleting the rule that says *"compute the result in SQL"*. Re-run the cells that build the agent, then repeat Exercise 2. Does the agent's behavior or accuracy change?

### Your own questions
Use the cell below to ask anything about the CRM data, for example about invoices, support tickets, or marketing campaigns.

In [ ]:
result = ask("...")  # replace ... with your question

## Part 8. Limitations and key takeaways

### Where text-to-SQL agents go wrong
- **Plausible but wrong answers.** The agent can write a query that runs without error but answers a slightly different question, for example by filtering on the wrong column or forgetting a join. The answer still *sounds* right. Always read the SQL.
- **Ambiguous questions.** Words such as "best", "recent", or "active" must be turned into exact conditions, and the agent may pick a meaning you did not intend.
- **Run-to-run variation.** Even with `temperature=0`, small changes in wording can lead to a different query. Smaller models make more mistakes than larger ones.
- **Large databases.** This database has 36 tables. Real company databases can have thousands, too many for the model to read. In practice you restrict the agent to the relevant tables, e.g. `SQLDatabase.from_uri(..., include_tables=[...])`.
- **Speed.** Every Thought/Action/Observation round is a full LLM call, so one question can take much longer than running the SQL directly.

### Good practices
- Give the agent a **read-only** connection (or a database user that can only read). Treat prompt rules as a helpful extra layer, never as the only protection.
- Keep the agent's **intermediate steps** and show the SQL to users, so answers can be checked.
- Build a small **test set** of questions with known answers (like the exercises above) and re-run it whenever you change the model or the prompt.
- Prefer a **locally hosted model** when the data is private, so it never leaves your system.

### Key takeaways
1. A text-to-SQL agent is an LLM in a loop that uses tools to explore the database, write a query, run it, and fix its own errors.
2. The system prompt and the tool descriptions are how you steer the agent's behavior.
3. The agent is a helpful assistant, not an oracle: verify its SQL and its answers.

### Going further
- `langgraph_agent.py` in this repository builds a similar agent with LangGraph, LangChain's newer framework for agents.
- `streamlit_app.py` puts a chat-style web interface on top of the agent.
- `Chinook.db` is a second sample database (a digital music store) for practicing with different questions.